# Jacobi



In [6]:
import time
import sys
from pathlib import Path
import numpy as np
from scipy.sparse import csr_matrix, issparse

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside Numerical_Linear_Algebra_Algorithms")
sys.path.insert(0, str(ROOT))
from common import loader, data_path, lu_solve, relative_residual
from Iterative_Methods.jacobi import jacobi


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


## 3×3


In [7]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = jacobi(A, b)
result, seconds = timed(jacobi, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
assert relative_residual(A, x, b) < 1e-8


Seconds: 0.0005692080012522638
{'iterations': 27, 'residual': 7.026334836242635e-09, 'converged': True}
Relative residual: 7.026334836242635e-09
Relative solution error: 7.484994579989294e-09


## SuiteSparse

Change MATRIX to an existing file. bcsstk09 is 1,083×1,083; bcsstk17 is 10,974×10,974; west1505 is nonsymmetric and requires pivoting. No 10^5 matrix exists in this repo. Keep CSR storage for larger tests.


In [8]:
MATRIX = "bcsstk09"
MAT_PATH = None
A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=False)
print("Matrix:", MATRIX, "shape:", A.shape)
x_true = np.ones(A.shape[0])
b = A @ x_true


Matrix: bcsstk09 shape: (1083, 1083)


In [9]:
result, seconds = timed(jacobi, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))


Seconds: 0.15852224998525344
{'iterations': 10000, 'residual': 4.9106471066607285e-05, 'converged': False}
Relative residual: 4.9106471066607285e-05
Relative solution error: 0.06521026058561424
